In [ ]:
## Vegetation Indices (VIs)

import os
import numpy as np
import rasterio

# Band order #
# Band 1 = Blue
# Band 2 = Green
# Band 3 = Red
# Band 4 = Red Edge
# Band 5 = NIR
# Band 6 = ignored

INPUT_DIR = r"D:/Kiran/Harz/03_Micasense"
OUTPUT_DIR = r"D:/Kiran/Harz_Results/QGIS_Harz"

OUT_NDVI  = os.path.join(OUTPUT_DIR, "NDVIs") 
OUT_SAVI  = os.path.join(OUTPUT_DIR, "SAVIs")    
OUT_GNDVI = os.path.join(OUTPUT_DIR, "GNDVIs") 
OUT_NDRE  = os.path.join(OUTPUT_DIR, "NDREs")   
OUT_VARI  = os.path.join(OUTPUT_DIR, "VARIs")   
OUT_EVI   = os.path.join(OUTPUT_DIR, "EVIs")    
OUT_MSAVI = os.path.join(OUTPUT_DIR, "MSAVIs")  
OUT_CIG   = os.path.join(OUTPUT_DIR, "CIgs")  

SITES = {
    # "SITE_CODE": r"FULL_PATH_TO_MULTIBAND_TIF"
    "WW":   r"D:/Kiran/Harz/03_Micasense/NPHarz_Altum_Wolfswarte_Orthomosaic_EPSG32632.tif",
    "GOS":  r"D:/Kiran/Harz/03_Micasense/NPHarz_Altum_Grane_Orthomosaic_EPSG32632.tif",
    "MGH1": r"D:/Kiran/Harz/03_Micasense/NPHarz_Altum_MagdeburgerHutte01_Orthomosaic_EPSG3632.tif",
    "MGH2": r"D:/Kiran/Harz/03_Micasense/NPHarz_Altum_MagdeburgerHutte02_Orthomosaic_EPSG32632.tif",
    "GB":   r"D:/Kiran/Harz/03_Micasense/NPHarz_Altum_GelberBrink_Orthomosaic_EPSG32632_georeferenced.tif",
    "QB":   r"D:/Kiran/Harz/03_Micasense/NPHarz_Altum_Quesenbank_Orthomosaic_EPSG32632_georeferenced.tif",
    "CHH":  r"D:/Kiran/Harz/03_Micasense/NPHarz_Altum_ChristianenHauss_Orthomosaic_EPSG32632_georeferenced.tif",
    "MK":   r"D:/Kiran/Harz/03_Micasense/NPHarz_Altum_Molkenhausschause_Orthomosaic_EPSG32632_georeferenced.tif",
}

# QGIS-STYLE VIs (Same as QGIS Raster Calculator) #
# Use the raster's internal mask (dataset_mask) so outputs do not create a "new background"; they follow the input footprint like QGIS does #

def safe_div(numer: np.ndarray, denom: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    """
    Safe division that avoids divide-by-zero.
    Wherever denom is ~0, returns NaN.
    """
    denom_safe = np.where(np.abs(denom) < eps, np.nan, denom)  # replace tiny denom with NaN
    return numer / denom_safe                                   # normal division elsewhere

def write_singleband(out_path: str, data: np.ndarray, profile: dict, mask: np.ndarray, nodata_val: float = 65535.0) -> None:
    """
    Write one-band float GeoTIFF preserving CRS/transform/shape,
    AND write the same mask as the input so the raster footprint matches exactly QGIS style.
    Converts NaNs to nodata value for QGIS friendliness.
    """
    prof = profile.copy()                      # copy georeferencing + raster metadata
    prof["driver"] = "GTiff"
    prof.update(
        count=1,                               # single band output
        dtype="float32",                       
        nodata=nodata_val,                    
        compress="deflate",                    
        predictor=2                           
    )

    out = data.astype(np.float32)              
    out = np.where(np.isfinite(out), out, nodata_val).astype(np.float32)  # NaN -> nodata

    out_mask = np.where(mask > 0, 255, 0).astype(np.uint8) # mask must be uint8 with 0 as transparent and 255 as valid 

    with rasterio.open(out_path, "w", **prof) as dst:
        dst.write(out, 1)                      # write to band 1
        dst.write_mask(out_mask)               # removes the background like in QGIS

def compute_vis(blue: np.ndarray,
                green: np.ndarray,
                red: np.ndarray,
                rededge: np.ndarray,
                nir: np.ndarray):
    """
    Computes 8 vegetation indices:
    NDRE, VARI, EVI, CIgreen, MSAVI, NDVI, SAVI, GNDVI
    """
    
    # NDRE = (NIR - RedEdge) / (NIR + RedEdge)
    
    ndre = safe_div(nir - rededge, nir + rededge)

    # VARI = (Green - Red) / (Green + Red - Blue)
   
    vari = safe_div(green - red, green + red - blue)

    # EVI = 2.5 * (NIR - Red) / (NIR + 6*Red - 7.5*Blue + 1)
  
    evi = 2.5 * safe_div((nir - red), (nir + 6.0 * red - 7.5 * blue + 1.0))

    # CIgreen = (NIR / Green) - 1

    cig = safe_div(nir, green) - 1.0

    # MSAVI = (2*NIR + 1 - sqrt((2*NIR + 1)^2 - 8*(NIR - Red))) / 2
    
    term = (2.0 * nir + 1.0)                 
    inside = term**2 - 8.0 * (nir - red)      # discriminant inside sqrt
    inside = np.where(inside < 0, np.nan, inside)  # numerical safety, if inside becomes negative, set to NaN
    msavi = (term - np.sqrt(inside)) / 2.0

   
    # NDVI = (NIR - Red) / (NIR + Red)
  
    ndvi = safe_div(nir - red, nir + red)


    # SAVI = ((NIR - Red) / (NIR + Red + L)) * (1 + L)
    # Common choice: L = 0.5
    L = 0.5
    savi = safe_div((nir - red), (nir + red + L)) * (1.0 + L)

    # GNDVI = (NIR - Green) / (NIR + Green)

    gndvi = safe_div(nir - green, nir + green)

    return ndre, vari, evi, cig, msavi, ndvi, savi, gndvi

# Main loop: process each site file and write outputs #
processed = 0  # count how many sites were successfully processed

for code, in_path in SITES.items():

    # check file exists #
    if not os.path.exists(in_path):
        print(f"[MISSING] {code}: {in_path}")
        continue

    try:
        # open the multiband raster #
        with rasterio.open(in_path) as src:

            # keep profile for writing outputs with same CRS/transform #
            profile = src.profile

            # read bands 1..5 in required order #
            blue    = src.read(1).astype(np.float32) 
            green   = src.read(2).astype(np.float32)  
            red     = src.read(3).astype(np.float32)  
            rededge = src.read(4).astype(np.float32)  
            nir     = src.read(5).astype(np.float32) 

            # QGIS-like background handling #
            # remove 65535 background (same as QGIS Additional NoData) #
            nodata_val = 65535   # max value from band 

            mask_outside = (
                (blue == nodata_val) |
                (green == nodata_val) |
                (red == nodata_val) |
                (rededge == nodata_val) |
                (nir == nodata_val)
            )

            footprint = np.where(mask_outside, 0, 255).astype(np.uint8)
            
            # set outside pixels to NaN so all VIs become nodata there #
            blue    = np.where(mask_outside, np.nan, blue)
            green   = np.where(mask_outside, np.nan, green)
            red     = np.where(mask_outside, np.nan, red)
            rededge = np.where(mask_outside, np.nan, rededge)
            nir     = np.where(mask_outside, np.nan, nir)
            
            # compute all vegetation indices #
            ndre, vari, evi, cig, msavi, ndvi, savi, gndvi = compute_vis(
                blue, green, red, rededge, nir,
            )
             
            
            # OUTPUT FILENAMES #
            ndre_name  = f"NDRE_{code}.tif"
            vari_name  = f"VARI_{code}.tif"
            evi_name   = f"EVI_{code}.tif"
            msavi_name = f"MSAVI_{code}.tif"
            cig_name   = f"CIgs_{code}.tif"
            ndvi_name  = f"NDVI_{code}.tif"
            savi_name  = f"SAVI_{code}.tif"
            gndvi_name = f"GNDVI_{code}.tif"

            # build full output paths #
            out_ndre  = os.path.join(OUT_NDRE,  ndre_name)
            out_vari  = os.path.join(OUT_VARI,  vari_name)
            out_evi   = os.path.join(OUT_EVI,   evi_name)
            out_msavi = os.path.join(OUT_MSAVI, msavi_name)
            out_cig   = os.path.join(OUT_CIG,   cig_name)
            out_ndvi  = os.path.join(OUT_NDVI,  ndvi_name)
            out_savi  = os.path.join(OUT_SAVI,  savi_name)
            out_gndvi = os.path.join(OUT_GNDVI, gndvi_name)

            # write GeoTIFFs (each is one-band float raster) #
            write_singleband(out_ndre,  ndre,  profile, footprint)
            write_singleband(out_vari,  vari,  profile, footprint)
            write_singleband(out_evi,   evi,   profile, footprint)
            write_singleband(out_msavi, msavi, profile, footprint)
            write_singleband(out_cig,   cig,   profile, footprint)
            write_singleband(out_ndvi,  ndvi,  profile, footprint)
            write_singleband(out_savi,  savi,  profile, footprint)
            write_singleband(out_gndvi, gndvi, profile, footprint)

            processed += 1
            print(f"[OK] {code}: 8 VIs created")

    except Exception as e:
        print(f"[ERROR] {code}: {in_path}\n   -> {e}")

# FINAL SUMMARY #

print("\n========== DONE ==========")
print(f"Processed sites: {processed}/{len(SITES)}")
print(f"Outputs saved under: {OUTPUT_DIR}")
